# Class-center crop

Same transforms as `train_proj.jl`. `VocRandomCrop` scales and cuts a uniform window. `VocClassCrop` scales and cuts on a foreground class, falling back to a uniform window when the mask has none.

In [ ]:
using Pkg
Pkg.activate("../")
using Images
using MLUtils
using DataFrames
using CSV
using Random
using DataAugmentation
using OffsetArrays
include("DataAugmentationCrops.jl")

In [ ]:
function get_image(path)
    return Images.load(expanduser(path)) .|> RGB{N0f8}
end

function get_mask(path)
    return Images.load(expanduser(path))
end

pascal_ids(mask::AbstractMatrix{<:Integer}) = mask
pascal_ids(mask) = mask.index

function wrap_pair(img, mask, ids::Function)
    return Image(img), MaskMulti(mask_ids(mask, ids) .+ 1, 1:256)
end

function unwrap_pair(img_wrap, mask_wrap)
    img = OffsetArrays.no_offset_view(img_wrap.data) .|> RGB{N0f8}
    mask = OffsetArrays.no_offset_view((mask_wrap.data .- 1) .|> Int16)
    return img, mask
end

In [ ]:
const IGNORE = 0xff

function colorize(mask)
    palette = RGB{N0f8}[
        RGB{N0f8}(0.00, 0.00, 0.00),
        RGB{N0f8}(0.50, 0.00, 0.00), RGB{N0f8}(0.00, 0.50, 0.00), RGB{N0f8}(0.50, 0.50, 0.00),
        RGB{N0f8}(0.00, 0.00, 0.50), RGB{N0f8}(0.50, 0.00, 0.50), RGB{N0f8}(0.00, 0.50, 0.50),
        RGB{N0f8}(0.50, 0.50, 0.50), RGB{N0f8}(0.75, 0.25, 0.25), RGB{N0f8}(0.25, 0.75, 0.25),
        RGB{N0f8}(0.25, 0.25, 0.75), RGB{N0f8}(0.75, 0.75, 0.25), RGB{N0f8}(0.75, 0.25, 0.75),
        RGB{N0f8}(0.25, 0.75, 0.75), RGB{N0f8}(0.75, 0.50, 0.25), RGB{N0f8}(0.50, 0.25, 0.75),
        RGB{N0f8}(0.25, 0.50, 0.75), RGB{N0f8}(0.90, 0.60, 0.60), RGB{N0f8}(0.60, 0.90, 0.60),
        RGB{N0f8}(0.60, 0.60, 0.90), RGB{N0f8}(0.90, 0.90, 0.60),
    ]
    return map(mask) do v
        v == IGNORE ? RGB{N0f8}(0, 0, 0) : palette[Int(v) + 1]
    end
end

function draw_row(img, mask, tfm)
    pad = RGB{N0f8}(1, 1, 1)
    tiles = map(1:4) do _
        wrapped = wrap_pair(img, mask, pascal_ids)
        ci, cm = unwrap_pair(apply(tfm, wrapped)...)
        mosaicview(ci, colorize(cm); nrow=1, npad=4, fillvalue=pad)
    end
    return mosaicview(tiles...; nrow=1, npad=12, fillvalue=pad)
end

In [ ]:
datasetfolder = "../dataset/"
df = CSV.read(expanduser(datasetfolder) * "dftrain.csv", DataFrame)
Random.seed!(1234)
N = size(df, 1)
train_idx, _ = MLUtils.splitobs(1:N, at=0.7, shuffle=true)
dftrain = df[train_idx, :]
N = size(dftrain, 1)

In [ ]:
k = rand(1:N)
row = dftrain[k, :]
img = get_image(row.X)
mask = get_mask(row.y)
println("sample $k   image $(size(img))   classes $(sort(unique(pascal_ids(mask))))")
mosaicview(img, mask; nrow=1)

Four random crops. Short side is drawn from `320:500`, then a uniform 256 window.

In [ ]:
random_crop = RandomCentricCrop(short_sides=320:500, crop=256)
draw_row(img, mask, random_crop)

Four class-center crops. A foreground class is drawn, then a pixel of that class. A mask with no foreground falls back to the uniform window.

In [ ]:
class_crop = ClassCentricCrop(short_sides=320:500, crop=256, ids=pascal_ids)
draw_row(img, mask, class_crop)